In [2]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.getenv("NASA_API_KEY")
BASE_URL = "https://api.nasa.gov"

response = requests.get(
    f"{BASE_URL}/planetary/apod",
    params = {
        "api_key": API_KEY,
        "date" : "2025-06-01"
    }
)

print(response.status_code)
print(json.dumps(response.json()))

200
{"copyright": "Domingo Pestana", "date": "2025-06-01", "explanation": "What's happening to this spiral galaxy? Although details remain uncertain, it surely has to do with an ongoing battle with its smaller galactic neighbor. The featured galaxy is labelled UGC 1810 by itself, but together with its collisional partner is known as Arp 273. The overall shape of UGC 1810 -- in particular its blue outer ring -- is likely a result of wild and violent gravitational interactions. This ring's blue color is caused by massive stars that are blue hot and have formed only in the past few million years.  The inner galaxy appears older, redder, and threaded with cool filamentary dust.  A few bright stars appear well in the foreground, unrelated to UGC 1810, while several galaxies are visible well in the background.  Arp 273 lies about 300 million light years away toward the constellation of Andromeda.  Quite likely, UGC 1810 will devour its galactic sidekick over the next billion years and settle

In [3]:
# check all the keys
data = response.json()
print(data.keys())

dict_keys(['copyright', 'date', 'explanation', 'hdurl', 'media_type', 'service_version', 'title', 'url'])


In [4]:
# print each field and its type
for key, value in data.items():
    print(f"{key:20} {type(value).__name__:10} {str(value)[:60]}")

copyright            str        Domingo Pestana
date                 str        2025-06-01
explanation          str        What's happening to this spiral galaxy? Although details rem
hdurl                str        https://apod.nasa.gov/apod/image/2506/Arp273Main_HubblePesta
media_type           str        image
service_version      str        v1
title                str        UGC 1810: Wildly Interacting Galaxy from Hubble
url                  str        https://apod.nasa.gov/apod/image/2506/Arp273Main_HubblePesta


copyright        → nice to have, attribution
date             → essential, primary key
explanation      → GOLD for RAG — rich astronomy text
hdurl            → high res image URL, keep it
media_type       → important — can be 'image' or 'video'
service_version  → internal API versioning, drop it
title            → essential, good for search
url              → standard res image URL, keep it

In [6]:
# pull a few days and see if copyright is always present
import pandas as pd

responses = []
for date in ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"]:
    r = requests.get(
        f"{BASE_URL}/planetary/apod",
        params={"api_key": API_KEY, "date": date}
    )
responses.append(r.json())

df = pd.DataFrame(responses)
print(df[["date", "title", "media_type", "copyright"]].to_string())
print("\nMissing copyright:", df["copyright"].isna().sum())

         date                        title media_type                                                                                  copyright
0  2026-04-01  The Claw and Bubble Nebulae      image  Richard Whitehead\n\nText:\nKeighley Rockcliffe  \n(NASA\nGSFC, \nUMBC CSST, \nCRESST II)

Missing copyright: 0


In [9]:
# test pulling a range
response = requests.get(
    f"{BASE_URL}/planetary/apod",
    params={
        "api_key": API_KEY,
        "start_date": "2026-05-01",
        "end_date": "2026-05-07",
    }
)
data = response.json()
print(type(data))
print(f"Entries returned: {len(data)}")
print(json.dumps(data[0], indent=2))

print(response.status_code)
print(response.text)

<class 'list'>
Entries returned: 7
{
  "copyright": "Chuck Ayoub",
  "date": "2026-05-01",
  "explanation": "Near the heart of the Virgo Galaxy Cluster, a string of galaxies known as Markarian's Chain stretches across this telescopic field of view. Anchored in the frame at bottom right by prominent lenticular galaxies, M84 (bottom) and M86, you can follow the chain's gentle arc up and toward the left. Near center you'll spot the pair of interacting galaxies NGC 4438 and NGC 4435, known to some as Markarian's Eyes. An estimated 50 million light-years distant, the Virgo Cluster itself is the nearest galaxy cluster. With up to about 2,000 member galaxies, it has a noticeable gravitational influence on our own Local Group of Galaxies. Within the Virgo Cluster at least seven galaxies in Markarian's Chain  appear to move coherently, while others may appear to be part of the chain by chance.",
  "hdurl": "https://apod.nasa.gov/apod/image/2605/M84andM86.png",
  "media_type": "image",
  "servic

copyright is missing on some entries — May 2nd, 3rd, 4th, 7th have no copyright field. Those are NASA/public domain images. So copyright is optional, unlike what the small sample suggested earlier.
media_type can be video — May 4th and 7th are videos, not images. The url points to .mp4 instead of .jpg. Important for the pipeline — we still want to keep these for RAG since the explanation text is just as rich.